# Main Pipeline: Fluorinated Amide Additives for Aqueous Li-ion Batteries

One-click Colab workflow for ORCA + OPI with 10 unique job IDs and Drive-safe storage.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/My Drive/DFT_Project'


In [ ]:
!bash /content/drive/My\ Drive/DFT_Project/repo/src/install_orca.sh


## Job orchestration

Run each job in a separate Colab instance/tab with a unique `JOB_ID` to avoid Drive file locks.


In [ ]:
import os
from pathlib import Path

JOB_IDS = [f'Job_{i:02d}' for i in range(1, 11)]
for job_id in JOB_IDS:
    Path(f'{PROJECT_ROOT}/{job_id}').mkdir(parents=True, exist_ok=True)
print('Prepared 10 unique subfolders')


In [ ]:
from opi.core import Calculator

keyword_line = '! PBEh-3c Opt RIJONX'

def build_input(job_id: str, xyz_path: str):
    calc = Calculator(
        method='PBEh-3c',
        basis=None,
        keywords=keyword_line,
        charge=0,
        multiplicity=1,
    )
    inp_text = calc.get_input(xyz=xyz_path)
    with open(f'{PROJECT_ROOT}/{job_id}/job.inp', 'w') as f:
        f.write(inp_text + '\n%output\n Print[ P_Json ] 1\nend\n')



In [ ]:
# Example single-instance launcher
import subprocess

JOB_ID = 'Job_01'  # Change this in each parallel Colab session
workdir = f'{PROJECT_ROOT}/{JOB_ID}'
xyz = '/content/drive/My Drive/DFT_Project/repo/inputs/TFMA.xyz'
build_input(JOB_ID, xyz)
proc = subprocess.Popen(['orca', f'{workdir}/job.inp'], cwd=workdir)
print('PID:', proc.pid)


In [ ]:
# Health-check monitor (every 10 min)
!python /content/drive/My\ Drive/DFT_Project/repo/src/monitor.py \
  --workdir "$workdir" \
  --base-input /content/drive/My\ Drive/DFT_Project/repo/templates/base_job.inp.template \
  --json-output "$workdir/job_property.json" \
  --xyz-seed "$xyz" \
  --poll-seconds 600


In [ ]:
# Post-processing + manuscript outputs
!python /content/drive/My\ Drive/DFT_Project/repo/src/analysis_XAI.py \
  --jobs-root "$PROJECT_ROOT" \
  --output-dir "$PROJECT_ROOT/analysis"


## Optional AI drafting integration

- Upload `results_summary.csv` to Consensus AI / Elicit / Scholarcy for top-paper retrieval.
- Paste the synthesized bullet points back into manuscript sections for final curation.
